# Kepler Field M-dwarf crossmatch (McQuillan+2013) → Gaia DR3 → 2MASS → Mann 2019 mass

Pulls McQuillan, Aigrain & Mazeh 2013 (MNRAS 432, 1203) — ~1570 Kepler M dwarfs with measured rotation periods — and joins them to the same Gaia/2MASS/mass pipeline as the training set.

Output: `cf_data/mcquillan_kepler_mdwarfs.csv` with the same column schema as `training_stars.csv` (minus age columns). Use for model-vs-field comparison.

## 1. Setup

In [15]:
from pathlib import Path
import sys, time
import numpy as np
import pandas as pd
import requests

from astroquery.vizier import Vizier
from astroquery.gaia    import Gaia

CF_DATA = Path('cf_data')
sys.path.insert(0, 'Mann_2019')  # for mk_mass

Vizier.ROW_LIMIT = -1   # no row cap
Vizier.TIMEOUT   = 300

def chunk_list(lst, n=500):
    for i in range(0, len(lst), n):
        yield lst[i:i+n]

## 2. Pull McQuillan+2013 from VizieR (J/MNRAS/432/1203)

In [16]:
# McQuillan 2013 = J/MNRAS/432/1203
# table2 is the main detected-rotation catalog (has 'Per' column)
v = Vizier(columns=['**'])
v.ROW_LIMIT = -1
cats = v.get_catalogs('J/MNRAS/432/1203')
print('Tables returned:')
for k, t in enumerate(cats):
    print(f'  [{k}] {t.meta.get("name", "?")}  rows={len(t)}  cols={t.colnames}')

mcq = cats[0].to_pandas()
print(f'\nMcQuillan rows: {len(mcq)}')
mcq.head()

Tables returned:
  [0] J/MNRAS/432/1203/table2  rows=1570  cols=['recno', 'KIC', 'Teff', 'logg', 'f_logg', 'Mass', 'Per', 'e_Per', 'Amp', 'Flag', '_RA', '_DE']
  [1] J/MNRAS/432/1203/table3  rows=121  cols=['recno', 'KIC', 'Mass', 'logg', 'f_logg', 'Teff', 'Amp', '_RA', '_DE']
  [2] J/MNRAS/432/1203/table4  rows=39  cols=['recno', 'KIC', 'Mass', 'logg', 'f_logg', 'Teff', 'Amp', '_RA', '_DE']
  [3] J/MNRAS/432/1203/table5  rows=753  cols=['recno', 'KIC', 'Mass', 'logg', 'f_logg', 'Teff', 'Amp', '_RA', '_DE']

McQuillan rows: 1570


,recno,KIC,Teff,logg,f_logg,Mass,Per,e_Per,Amp,Flag,_RA,_DE
0,1,1162635,3899,4.62,,0.5037,15.509000,0.064,10.700000,,291.33484,36.806438
1,2,1430893,3956,4.41,,0.5260,17.143999,0.046,10.400000,,291.08667,37.072559
2,3,1572802,3990,4.48,,0.5394,0.368000,0.000,74.800003,PB,291.29581,37.155861
3,4,1721911,3833,4.58,,0.4781,28.403000,0.394,3.900000,,291.62067,37.227280
4,5,1866535,3878,4.50,,0.4955,25.052000,0.136,4.000000,,291.02719,37.352089


In [17]:
# Standardise column names. Actual McQuillan 2013 columns:
#   recno, KIC, Teff, logg, f_logg, Mass, Per, e_Per, Amp, Flag, _RA, _DE
rename = {'KIC': 'kic_id', 'Per': 'prot_days', 'e_Per': 'prot_err_days',
          '_RA': 'ra', '_DE': 'dec', 'Teff': 'teff_k_mcq', 'Mass': 'mass_mcq'}
rename = {k: v for k, v in rename.items() if k in mcq.columns}
mcq = mcq.rename(columns=rename)

mcq['kic_id']       = mcq['kic_id'].astype('Int64')
mcq['source_paper'] = 'McQuillan2013'

print(f'After rename: {list(mcq.columns)}')
print(f'KIC unique: {mcq["kic_id"].nunique()}  |  P_rot range: {mcq["prot_days"].min():.2f} - {mcq["prot_days"].max():.2f} d')

After rename: ['recno', 'kic_id', 'teff_k_mcq', 'logg', 'f_logg', 'mass_mcq', 'prot_days', 'prot_err_days', 'Amp', 'Flag', 'ra', 'dec', 'source_paper']
KIC unique: 1570  |  P_rot range: 0.37 - 69.68 d


## 3. McQuillan → Gaia DR3 crossmatch (CDS XMatch, position-based)

3 arcsec cone match against `vizier:I/355/gaiadr3`. Keep nearest match per KIC.

In [18]:
from astroquery.xmatch import XMatch
from astropy.table import Table
import astropy.units as u

src = mcq[['kic_id', 'ra', 'dec']].dropna(subset=['ra', 'dec']).copy()
print(f'Crossmatching {len(src)} McQuillan positions against Gaia DR3 (3 arcsec, chunked)...')

def xmatch_chunk(chunk_df, retries=3, delay=10):
    for attempt in range(1, retries + 1):
        try:
            t = Table.from_pandas(chunk_df)
            res = XMatch.query(cat1=t, cat2='vizier:I/355/gaiadr3',
                               max_distance=3 * u.arcsec,
                               colRA1='ra', colDec1='dec',
                               colRA2='RA_ICRS', colDec2='DE_ICRS')
            return res.to_pandas()
        except Exception as e:
            if attempt < retries:
                print(f'    attempt {attempt} failed ({type(e).__name__}), retrying in {delay}s...')
                time.sleep(delay)
            else:
                print(f'    all {retries} attempts failed; skipping chunk')
                return pd.DataFrame()

parts = []
chunks = list(chunk_list(src.index.tolist(), n=300))
for i, idx_chunk in enumerate(chunks, 1):
    print(f'  chunk {i}/{len(chunks)}: {len(idx_chunk)} positions')
    parts.append(xmatch_chunk(src.loc[idx_chunk]))
    time.sleep(1)

xm = pd.concat([p for p in parts if len(p)], ignore_index=True)
print(f'  raw matches returned: {len(xm)}')

# Best match per kic_id (smallest angular distance)
xm = xm.sort_values('angDist').drop_duplicates('kic_id', keep='first')
xm['kic_id']    = xm['kic_id'].astype('Int64')
src_col = 'DR3Name' if 'DR3Name' in xm.columns else ('Source' if 'Source' in xm.columns else None)
if src_col is None:
    raise RuntimeError(f'No Gaia ID column found in XMatch output. Columns: {xm.columns.tolist()}')
if src_col == 'DR3Name':
    xm['source_id'] = xm['DR3Name'].astype(str).str.replace('Gaia DR3 ', '', regex=False).astype('Int64')
else:
    xm['source_id'] = xm['Source'].astype('Int64')

mcq_g = mcq.merge(xm[['kic_id', 'source_id', 'angDist']], on='kic_id', how='left')
matched = mcq_g['source_id'].notna().sum()
print(f'McQuillan → Gaia DR3 matched: {matched} / {len(mcq_g)}')

Crossmatching 1570 McQuillan positions against Gaia DR3 (3 arcsec, chunked)...
  chunk 1/6: 300 positions
  chunk 2/6: 300 positions
  chunk 3/6: 300 positions
  chunk 4/6: 300 positions
  chunk 5/6: 300 positions
  chunk 6/6: 70 positions
  raw matches returned: 1825
McQuillan → Gaia DR3 matched: 1559 / 1570


## 4. Query Gaia DR3 source data

In [19]:
def query_gaia_source(source_ids):
    ids_str = ','.join(str(x) for x in source_ids)
    q = f"""
        SELECT source_id, phot_g_mean_mag, bp_rp, ruwe, parallax, parallax_error, ra, dec
        FROM gaiadr3.gaia_source
        WHERE source_id IN ({ids_str})
    """
    return Gaia.launch_job(q).get_results().to_pandas()

def query_gaia_ap(source_ids):
    ids_str = ','.join(str(x) for x in source_ids)
    q = f"""
        SELECT source_id, ebpminrp_gspphot
        FROM gaiadr3.astrophysical_parameters
        WHERE source_id IN ({ids_str})
    """
    return Gaia.launch_job(q).get_results().to_pandas()

dr3_ids = mcq_g['source_id'].dropna().astype(int).unique().tolist()
print(f'Querying Gaia for {len(dr3_ids)} DR3 IDs...')

core_parts, ap_parts = [], []
for i, chunk in enumerate(chunk_list(dr3_ids, n=500), start=1):
    core_parts.append(query_gaia_source(chunk))
    ap_parts.append(query_gaia_ap(chunk))
    if i % 5 == 0:
        print(f'  chunk {i}: {sum(len(p) for p in core_parts)} rows so far')

gaia = pd.concat(core_parts, ignore_index=True).merge(
    pd.concat(ap_parts, ignore_index=True), how='left', on='source_id'
)
gaia['bp_rp_0']   = gaia['bp_rp'] - gaia['ebpminrp_gspphot']
gaia['source_id'] = gaia['source_id'].astype('Int64')
print(f'Gaia rows returned: {len(gaia)}')

Querying Gaia for 1559 DR3 IDs...
Gaia rows returned: 1559


## 5. 2MASS K-band photometry

In [20]:
# Step 1: source_id → 2MASS designation via gaiadr3.tmass_psc_xsc_best_neighbour
desig_parts = []
for i, chunk in enumerate(chunk_list(dr3_ids, n=500), start=1):
    ids_str = ','.join(str(x) for x in chunk)
    q = f"""
        SELECT source_id, original_ext_source_id AS twomass_id
        FROM gaiadr3.tmass_psc_xsc_best_neighbour
        WHERE source_id IN ({ids_str})
    """
    desig_parts.append(Gaia.launch_job(q).get_results().to_pandas())
    if i % 5 == 0:
        print(f'  designations chunk {i}')

desig = pd.concat(desig_parts, ignore_index=True)
desig['source_id']  = desig['source_id'].astype('Int64')
desig['twomass_id'] = desig['twomass_id'].str.strip()
print(f'Designations: {len(desig)} for {desig["source_id"].nunique()} sources')

# Step 2: 2MASS designation → Ks magnitude via CDS TAPVizieR II/246/out
VZ = 'http://tapvizier.cds.unistra.fr/TAPVizieR/tap/sync'
phot_parts = []
for i, chunk in enumerate(chunk_list(desig['twomass_id'].dropna().unique().tolist(), n=500), start=1):
    desig_str = ','.join(f"'{d}'" for d in chunk)
    q = f'SELECT "2MASS" AS twomass_id, "Kmag" AS k_m, "e_Kmag" AS k_cmsig FROM "II/246/out" WHERE "2MASS" IN ({desig_str})'
    r = requests.post(VZ, data={'REQUEST':'doQuery','LANG':'ADQL','FORMAT':'json','QUERY':q}, timeout=120)
    r.raise_for_status()
    d = r.json()
    cols = [c['name'] for c in d['metadata']]
    phot_parts.append(pd.DataFrame(d['data'], columns=cols))
    if i % 5 == 0:
        print(f'  photometry chunk {i}')
    time.sleep(0.5)

tmass_phot = pd.concat(phot_parts, ignore_index=True)
tmass_phot['twomass_id'] = tmass_phot['twomass_id'].str.strip()
tmass = desig.merge(tmass_phot, on='twomass_id', how='left')
print(f'2MASS rows with Ks: {tmass["k_m"].notna().sum()} / {len(tmass)}')

Designations: 1547 for 1547 sources


ConnectionError: ('Connection aborted.', ConnectionResetError(54, 'Connection reset by peer'))

## 6. Dereddening (Bayestar 2019 + Edenhofer 2023)

Inlined from `dereddening.ipynb` (main branch). Edenhofer-preferred; Bayestar fallback. Outputs per-star `A_Ks` ± `A_Ks_err`.

In [ ]:
# Dereddening — replicates dereddening.ipynb (main branch) for the McQuillan sample only
#   distance: Bailer-Jones EDR3 photogeometric (VizieR I/352/gedr3dis), fallback 1/parallax
#   dust:     Edenhofer 2023 integrated mean (preferred), Bayestar 2019 (fallback)
#   conversion: E -> A_V via map-specific E(B-V) factor, then A_Ks = 0.078 * A_V (Wang & Chen 2019)

import astropy.units as u
from astropy.coordinates import SkyCoord
from io import StringIO

from dustmaps.bayestar      import BayestarQuery
from dustmaps.edenhofer2023 import Edenhofer2023Query

# --- distances from Bailer-Jones EDR3 (VizieR I/352/gedr3dis) ---
def query_bailer_jones(source_ids, chunk=500):
    parts = []
    for i, c in enumerate(chunk_list(source_ids, n=chunk), 1):
        ids_str = ','.join(str(x) for x in c)
        q = ('SELECT Source, rgeo, b_rgeo, B_rgeo, rpgeo, b_rpgeo, B_rpgeo '
             f'FROM "I/352/gedr3dis" WHERE Source IN ({ids_str})')
        try:
            r = requests.post('https://tapvizier.cds.unistra.fr/TAPVizieR/tap/sync',
                              data={'REQUEST':'doQuery','LANG':'ADQL','FORMAT':'csv','QUERY':q},
                              timeout=120)
            r.raise_for_status()
            if r.text.strip():
                parts.append(pd.read_csv(StringIO(r.text)))
        except Exception as e:
            print(f'  BJ chunk {i} failed: {e}')
        time.sleep(0.5)
    if not parts:
        return pd.DataFrame(columns=['Source','rgeo','b_rgeo','B_rgeo','rpgeo','b_rpgeo','B_rpgeo'])
    return pd.concat(parts, ignore_index=True)

bj = query_bailer_jones(dr3_ids)
for c in ['rgeo','b_rgeo','B_rgeo','rpgeo','b_rpgeo','B_rpgeo']:
    bj[c] = pd.to_numeric(bj[c], errors='coerce')
bj['Source']      = bj['Source'].astype('Int64')
bj['dist_pc']     = np.where(np.isfinite(bj['rpgeo']), bj['rpgeo'], bj['rgeo'])
bj['dist_pc_err'] = np.where(np.isfinite(bj['rpgeo']),
                              (bj['B_rpgeo'] - bj['b_rpgeo']) / 2,
                              (bj['B_rgeo']  - bj['b_rgeo'])  / 2)
print(f'Bailer-Jones distances retrieved for {len(bj)} / {len(dr3_ids)} stars')

# Merge distance into gaia + fallback to 1/parallax
gaia_d = gaia.merge(bj[['Source','dist_pc','dist_pc_err']],
                    left_on='source_id', right_on='Source', how='left').drop(columns=['Source'])
fb = gaia_d['dist_pc'].isna() & gaia_d['parallax'].notna() & (gaia_d['parallax'] > 0)
gaia_d.loc[fb, 'dist_pc']     = 1000.0 / gaia_d.loc[fb, 'parallax']
gaia_d.loc[fb, 'dist_pc_err'] = 1000.0 * gaia_d.loc[fb, 'parallax_error'] / gaia_d.loc[fb, 'parallax']**2

# --- query dust maps ---
edenhofer = Edenhofer2023Query(load_samples=False, integrated=True)
bayestar  = BayestarQuery(max_samples=10)
N_DSAMP   = 10
A_KS_OVER_AV = 0.078   # Wang & Chen 2019
A_KS_OVER_AV_ERR = 0.004
Rv = 3.1

ext_rows = []
queryable = gaia_d.dropna(subset=['ra','dec','dist_pc']).reset_index(drop=True)
print(f'Querying dust maps for {len(queryable)} stars...')

for i, r in queryable.iterrows():
    if i % 200 == 0:
        print(f'  {i} / {len(queryable)}')
    d, derr = r.dist_pc, r.dist_pc_err
    if pd.notna(derr) and derr > 0:
        ds = np.clip(np.random.normal(d, derr, N_DSAMP), 10.0, None)
    else:
        ds = np.array([max(d, 10.0)])
    b_samps, e_samps = [], []
    for dd in ds:
        co = SkyCoord(r.ra * u.deg, r.dec * u.deg, distance=dd * u.pc, frame='icrs')
        try:
            E_b = bayestar(co, mode='samples', return_flags=True)
            if (E_b[1][0] & E_b[1][1]):
                b_samps.extend(E_b[0])
        except Exception:
            pass
        try:
            E_e = edenhofer(co, mode='mean')
            if np.isfinite(E_e):
                e_samps.append(float(E_e))
        except Exception:
            pass

    if e_samps:
        EBV, EBV_err, used = np.nanmedian(e_samps) * 0.829, np.nanstd(e_samps) * 0.829, 'Edenhofer'
    elif b_samps:
        EBV, EBV_err, used = np.nanmedian(b_samps) * 0.88,  np.nanstd(b_samps) * 0.88,  'Bayestar'
    else:
        EBV, EBV_err, used = 0.0, 0.0, 'none'

    A_V     = Rv * EBV
    A_V_err = Rv * EBV_err
    A_Ks    = A_V * A_KS_OVER_AV
    A_Ks_err = (A_Ks * np.sqrt((A_V_err/A_V)**2 + (A_KS_OVER_AV_ERR/A_KS_OVER_AV)**2)
                if A_V > 0 else 0.0)
    ext_rows.append({'source_id': int(r.source_id), 'A_Ks': A_Ks, 'A_Ks_err': A_Ks_err, 'dustmap_used': used})

ext_df = pd.DataFrame(ext_rows)
ext_df['source_id'] = ext_df['source_id'].astype('Int64')
print(f'\nExtinction summary:')
print(ext_df['dustmap_used'].value_counts())
print(ext_df['A_Ks'].describe())

## 7. Compute mass via Mann 2019

In [ ]:
import mk_mass

MANN_MKS_MIN = 4.5
MANN_MKS_MAX = 10.5

# Merge Gaia + 2MASS + extinction into McQuillan
mcq_full = mcq_g.merge(gaia, on='source_id', how='left', suffixes=('', '_gaia'))
mcq_full = mcq_full.merge(tmass[['source_id', 'twomass_id', 'k_m', 'k_cmsig']],
                           on='source_id', how='left')
mcq_full = mcq_full.merge(ext_df[['source_id', 'A_Ks', 'A_Ks_err', 'dustmap_used']],
                           on='source_id', how='left')

# A_Ks fallback to 0 for stars not in ext_df (e.g. missing coords or distance)
mcq_full['A_Ks']     = mcq_full['A_Ks'].fillna(0.0)
mcq_full['A_Ks_err'] = mcq_full['A_Ks_err'].fillna(0.0)
mcq_full['k_m_0']    = mcq_full['k_m'] - mcq_full['A_Ks']

def row_to_mass(row):
    k0   = row['k_m_0']
    plx  = row.get('parallax', np.nan)
    eplx = row.get('parallax_error', np.nan)
    ek   = row.get('k_cmsig', 0.0) or 0.0
    eAKs = row.get('A_Ks_err', 0.0) or 0.0
    if pd.isna(k0) or pd.isna(plx) or plx <= 0:
        return pd.Series([np.nan, np.nan, np.nan, None],
                          index=['M_Ks', 'mass_msun', 'mass_msun_err', 'flag_outside_mann_range'])
    d   = 1000.0 / plx
    M_K = k0 + 5 - 5 * np.log10(d)
    if M_K < MANN_MKS_MIN or M_K > MANN_MKS_MAX:
        return pd.Series([M_K, np.nan, np.nan, True],
                          index=['M_Ks', 'mass_msun', 'mass_msun_err', 'flag_outside_mann_range'])
    try:
        ed   = (1000.0 / plx**2) * eplx if pd.notna(eplx) else 0.0
        ek_t = np.sqrt(ek**2 + eAKs**2)   # combined K-band uncertainty
        result = mk_mass.posterior(K=k0, K_err=ek_t, dist=d, dist_err=ed)
        m   = float(np.median(result))
        sig = float(0.5 * (np.percentile(result, 84) - np.percentile(result, 16)))
        return pd.Series([M_K, m, sig, False],
                          index=['M_Ks', 'mass_msun', 'mass_msun_err', 'flag_outside_mann_range'])
    except Exception:
        return pd.Series([M_K, np.nan, np.nan, None],
                          index=['M_Ks', 'mass_msun', 'mass_msun_err', 'flag_outside_mann_range'])

extras = mcq_full.apply(row_to_mass, axis=1)
mcq_full = pd.concat([mcq_full, extras], axis=1)

# Match training_stars.csv schema for the asymmetric mass errors
mcq_full['mass_msun_err_lo'] = mcq_full['mass_msun_err']
mcq_full['mass_msun_err_hi'] = mcq_full['mass_msun_err']
mcq_full['log_prot']         = np.log10(mcq_full['prot_days'])
mcq_full['flag_high_ruwe']   = mcq_full['ruwe'].apply(lambda r: r >= 1.2 if pd.notna(r) else None)

print('=== Crossmatch summary ===')
print(f'McQuillan stars      : {len(mcq_full)}')
print(f'Matched to Gaia DR3  : {mcq_full["source_id"].notna().sum()}')
print(f'Matched to 2MASS Ks  : {mcq_full["k_m"].notna().sum()}')
print(f'Has A_Ks > 0         : {(mcq_full["A_Ks"] > 0).sum()}')
print(f'Mass computed        : {mcq_full["mass_msun"].notna().sum()}')
print(f'Outside Mann range   : {mcq_full["flag_outside_mann_range"].eq(True).sum()}')
print(f'Flag high RUWE       : {mcq_full["flag_high_ruwe"].sum()}')

## 8. Save

In [ ]:
out_cols = ['source_paper', 'kic_id', 'source_id', 'twomass_id', 'ra', 'dec',
            'prot_days', 'log_prot', 'parallax', 'parallax_error',
            'bp_rp', 'bp_rp_0', 'ebpminrp_gspphot', 'ruwe', 'phot_g_mean_mag',
            'k_m', 'k_cmsig', 'A_Ks', 'A_Ks_err', 'dustmap_used', 'k_m_0', 'M_Ks',
            'mass_msun', 'mass_msun_err_lo', 'mass_msun_err_hi',
            'flag_outside_mann_range', 'flag_high_ruwe']
out_cols = [c for c in out_cols if c in mcq_full.columns]

out = mcq_full[out_cols].copy()
out_path = CF_DATA / 'mcquillan_kepler_mdwarfs.csv'
out.to_csv(out_path, index=False)
print(f'Saved {len(out)} rows to {out_path}')
out.head()